### Drift-Diffusion One-Dimensional (1-D) Classical Approach
###### Gaussian distribution

###### 2026. 09. 03 : Seungho Chung, updated the entire codebse for python 3.14 version

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import Output, VBox, Button, Label, FloatSlider

# Simulation parameters
x_min, x_max = 0, 0.1  # cm
num_points = 501
x_vals = np.linspace(x_min, x_max, num_points)
num_particles = 250
mu = 500  # cm^2/Vs Mobility constant (example value)
kT = 0.0259  # Thermal energy (eV) at room temperature
D = mu * kT  # Diffusion coefficient (example value)

# UI controls
E_slider = FloatSlider(value=20.0, min=0.0, max=40.0, step=10, description='E-Field (V/cm):', style={'description_width': 'initial'})
length_slider = FloatSlider(value=0.05, min=0.01, max=0.1, step=0.01, description='Length (cm):', style={'description_width': 'initial'})
t_max_slider = FloatSlider(value=10, min=10, max=31, step=5, description='Simulation Time (us):', style={'description_width': 'initial'})
dt_slider = FloatSlider(value=0.1, min=0.01, max=0.1, step=0.01, description='Time step (us):', style={'description_width': 'initial'})
run_button = Button(description='Run', button_style='success')
status_label = Label('Status: Idle')

# Output area
out = Output()

# Probability distribution function (combining diffusion and drift due to electric field)
def p(x, t, E, D):
    # Shift due to electric field E and drift mobility mu * E * t
    drift = mu * E * t  # drift due to electric field
    return np.exp(-(x - mu * E * t)**2 / (4 * D * t)) / np.sqrt(4 * np.pi * D * t)

# Simulation runner
def run_simulation(E, t_max, length, dt, status_label):
    num_frames = int(t_max/dt)+1  # Number of frames in the animation

    status_label.value = 'Status: Running...'

    # Precompute all time steps for the simulation
    times = np.linspace(0.1e-6, t_max + 0.1e-6, num_frames)
    p_vals_over_time = np.array([p(x_vals, t, E, D) for t in times])
    p_at_length_over_time = np.array([p(length, t, E, D) for t in times])

    # Find maximum value and its time
    max_p = np.max(p_at_length_over_time)
    max_time = times[np.argmax(p_at_length_over_time)]

    # Find the time where the probability is 1/e of the max value
    threshold_p = max_p / np.exp(1)
    # Find the time corresponding to the threshold
    t1_index = np.where(p_at_length_over_time >= threshold_p)[0][0]
    t2_index = np.where(p_at_length_over_time >= threshold_p)[0][-1]

    threshold_time1 = times[t1_index]
    threshold_time2 = times[t2_index]

    with out:
        out.clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Initial plot (with t=0)
        p_vals = p_vals_over_time[0]
        line_dist, = ax1.plot(x_vals, p_vals, 'b-', lw=2)
        ax1.plot([length,length],[0,np.max(p_vals) * 1.2],'b--')
        ax1.set_xlim(x_min, 0.1)
        ax1.set_ylim(0, np.max(p_vals) * 1.2)
        ax1.set_xticks([0,length,0.1]); 
        B_text = ax1.text(length- 0.9e-6, 310, f'Contact B', color='green', ha='left', va='top')

        # Add initial time label
        time_text = ax1.text(0.95, 0.95, f'Time = {times[0]*1e6:.2f} us', transform=ax1.transAxes, 
                            ha='right', va='top', fontsize=12, color='black')
        ax1.set_xlabel('Position (cm)')
        ax1.set_ylabel(f'P(x)')

        # Second plot for tracking value at selected length
        line_time, = ax2.plot(times, p_at_length_over_time, 'r-', lw=2)  # Full line plot
        ax2.set_xlim(0, t_max)  # Time axis limit (seconds)
        ax2.set_ylim(0, np.max(p_vals) * 1.2)  # Probability axis limit
        ax2.set_xlabel('Time (s)')
        ax2.set_ylabel(f'P(x={length:.2f} cm, t)')

        # Plot a single point for P(x=length, t)
        point, = ax2.plot([], [], 'bo', ms=6)  # 'bo' is for blue circles

        # Plot maximum point and 1/e threshold point
        max_point, = ax2.plot(max_time, max_p, 'go', ms=5, label='Max P(x, t)')  # Green circle for max
        threshold_point1, = ax2.plot(threshold_time1, threshold_p, 'mo', ms=5, label='1/e Max P(x, t)')  # Magenta circle for 1/e
        threshold_point2, = ax2.plot(threshold_time2, threshold_p, 'mo', ms=5)  # Magenta circle for 1/e

        # Adjust y-values to ensure points are visible on the graph
        ax2.set_ylim(0, np.max(p_vals) * 1.2)

        # Add text labels at each point
        max_text = ax2.text(max_time - 0.9e-6, 100, f't_max = {max_time*1e6:.2f} μs', color='green', ha='left', va='bottom')
        threshold_text1 = ax2.text(threshold_time1 - 1.8e-6, 80, f't1 = {threshold_time1*1e6:.2f} μs', color='magenta', ha='left', va='top')
        threshold_text2 = ax2.text(threshold_time2 + 0.1e-6, 80, f't2 = {threshold_time2*1e6:.2f} μs', color='magenta', ha='left', va='top')

        # Update function for animation
        def update(frame):
            p_vals = p_vals_over_time[frame]
            time_text.set_text(f'Time = {times[frame]*1e6:.2f} us')
            line_dist.set_ydata(p_vals)

            # Track value at selected length and update point position
            p_length = p_at_length_over_time[frame]
            point.set_data([times[frame]], [p_at_length_over_time[frame]])   # Update the point position

            return line_dist, time_text, line_time, point, max_point, threshold_point1, threshold_point2, B_text, max_text, threshold_text1, threshold_text2

        ani = animation.FuncAnimation(fig, update, frames=num_frames, interval=50, blit=True)
        display(HTML(ani.to_jshtml()))
        plt.close(fig)

    status_label.value = 'Status: Idle'

# Bind run button
def on_run(b):
    if status_label.value == 'Status: Running...':
        return
    run_simulation(E_slider.value, t_max_slider.value*1e-6, length_slider.value, dt_slider.value*1e-6, status_label)

run_button.on_click(on_run)

# Display UI
display(VBox([E_slider, length_slider, t_max_slider, dt_slider, run_button, status_label, out]))


### 2025 Spring "EE211: Physical Electronics"

###### Update history
###### 2025. 05. 11 : Jeongwon Lee, KAIST Electrical Engineering, Initial implementation of Haynes-Shockley experiment

###### ref
###### - Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012)

### Inputs
###### E-field          : Applied electric field    (eV/cm)
###### Length           : Length between A and B    (cm)
###### Simulation time  : Total simulation time     (us)
###### Time step        : Time step dt              (us)